### **Install Applio**
If the runtime restarts, re-run the installation steps.

In [ ]:
# @title Setup runtime environment
from IPython.display import clear_output
import codecs

encoded_url = "uggcf://tvguho.pbz/VNUvfcnab/Nccyvb/"
decoded_url = codecs.decode(encoded_url, "rot_13")

repo_name_encoded = "Nccyvb"
repo_name = codecs.decode(repo_name_encoded, "rot_13")

LOGS_PATH = f"/content/{repo_name}/logs"
BACKUPS_PATH = f"/content/drive/MyDrive/{repo_name}Backup"

%cd /content
!git config --global advice.detachedHead false
!git clone {decoded_url} --branch 3.6.2 --single-branch
%cd {repo_name}
clear_output()

!apt update -y
!apt install -y portaudio19-dev

print("Installing requirements...")
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match
!uv pip install -q ngrok jupyter-ui-poll
!npm install -g -q localtunnel &> /dev/null

!python core.py "prerequisites" --models "True" --pretraineds_hifigan "True"
print("✅ Finished installing requirements!")


### **Start Applio**

In [ ]:
!wget "https://huggingface.co/raptarantino/cartideepvoice/resolve/main/cartideepvoice.zip" -O male_deep.zip

In [ ]:
# Unzip male deep voice
!unzip -o male_deep.zip -d male_deep

In [ ]:
!ls -R male_deep

In [ ]:
# Create folder if it doesn't exist
!mkdir -p /content/rvc/pretraineds/custom

# Move Carti deep model files
!mv male_deep/cartideepvoice/*.pth /content/rvc/pretraineds/custom/
!mv male_deep/cartideepvoice/*.index /content/rvc/pretraineds/custom/

# Move Luke Bryan model files
!mv luke_bryan/*.pth /content/rvc/pretraineds/custom/
!mv luke_bryan/*.index /content/rvc/pretraineds/custom/

In [ ]:
import json

# Path to the correct config file
config_path = "/content/Applio/assets/config.json"

# Load existing config
with open(config_path, "r") as f:
    config = json.load(f)

# Update the model paths to your Carti deep male voice
config["realtime"]["model_file"] = "/content/Applio/male_deep/cartideepvoice/added_IVF1933_Flat_nprobe_1_carti2023_v2.index"
config["realtime"]["index_file"] = "/content/Applio/male_deep/cartideepvoice/carti2023_e425_s22950.pth"

# Save the updated config
with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print("Applio config.json updated successfully!")

In [ ]:
# @title **Start server**
# @markdown  ### Choose a sharing method:
from IPython.display import clear_output

method = "gradio"  # @param ["gradio", "localtunnel", "ngrok"]
ngrok_token = "If you selected the 'ngrok' method, obtain your auth token here: https://dashboard.ngrok.com/get-started/your-authtoken" # @param {type:"string"}
tensorboard = True #@param {type: "boolean"}

%cd /content/{repo_name}
clear_output()

if tensorboard:
  %load_ext tensorboard
  %tensorboard --logdir logs --bind_all

match method:
  case 'gradio':
    !python app.py --listen --share --client
  case 'localtunnel':
    !echo Password IP: $(curl --silent https://ipv4.icanhazip.com)
    !echo
    !lt --port 6969 & python app.py --listen --client
  case 'ngrok':
    import ngrok
    ngrok.kill()
    listener = await ngrok.forward(6969, authtoken=ngrok_token)
    print(f"Ngrok URL: {listener.url()}")
    !python app.py --listen --client